## Managed Memory for Databricks Agents

### Installing Utilities and Libraries

In [ ]:
%pip install databricks-sdk==0.49.0 openai-agents==0.22.0 databricks-openai==0.17.1

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Generate the OAuth Token for Memory Creation Purposes

In [ ]:
import requests

workspace_host = "YOUR-DATABRICKS-HOSTNAME-GOES-HERE"

client_id = "YOUR-SVC-PRINCIPAL-CLIENT-GOES-HERE"
client_secret = "YOUR-SVC-PRINCIPAL-SECRET-GOES-HERE"

response = requests.post(
    f"{workspace_host}/oidc/v1/token",
    auth=(client_id, client_secret),
    data={
        "grant_type": "client_credentials",
        "scope": "all-apis"
    }
)

response.raise_for_status()

DATABRICKS_TOKEN = response.json()["access_token"]

print("OAuth token generated successfully")

### Create the Managed Memory Store

In [ ]:
import requests

url = f"{workspace_host}/api/2.1/unity-catalog/memory-stores"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "name": "sustainability_memory",
    "catalog_name": "YOUR-CATALOG-NAME-GOES-HERE",
    "schema_name": "YOUR-CATALOG-SCHEMA-NAME-GOES-HERE",
    "description": "Long-term memory for CarbonOps sustainability"
    "managers, facility targets, and reporting preferences"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print("Status Code:", response.status_code)
print(response.json())

### Configure the Memory Scope

In [ ]:
MEMORY_STORE = "YOUR-CATALOG-NAME.YOUR-SCHEMA-NAME.sustainability_memory"

# Partition for this demo user's memories
MEMORY_SCOPE = "carbonops-manager-001"

print("Memory Store:", MEMORY_STORE)
print("Memory Scope:", MEMORY_SCOPE)

### Store CarbonOps Sustainability Context

We'll store three independent memories: the manager's profile, the Ahmedabad facility's targets, and the manager's reporting preferences.

In [ ]:
memory_url = (
    f"{workspace_host}/api/2.1/unity-catalog/"
    f"memory-stores/{MEMORY_STORE}/entries"
)

memories = [
    {
        "path": "/memories/manager_profile.md",
        "description": "Sustainability manager profile and responsibilities",
        "contents": """
Alex Morgan is a Regional Sustainability Manager at CarbonOps.

Alex oversees manufacturing sustainability performance
across facilities in Ahmedabad, Pune, and Chennai.

Alex's priorities include reducing Scope 1 and Scope 2
emissions, improving carbon intensity, and meeting
corporate sustainability targets.
"""
    },
    {
        "path": "/memories/ahmedabad_facility.md",
        "description": "Ahmedabad factory emissions targets and priorities",
        "contents": """
The CarbonOps Ahmedabad manufacturing facility has
a monthly carbon emissions budget of 200 tCO2e.

Its carbon intensity target is 8 kg CO2e per unit.

The facility's main decarbonization priorities are
reducing diesel consumption, improving electricity
efficiency, and increasing renewable electricity usage.

For demonstration purposes, its latest reported
emissions are 180 tCO2e and production is 25,000 units.
"""
    },
    {
        "path": "/memories/reporting_preferences.md",
        "description": "Preferred sustainability reporting format",
        "contents": """
Alex prefers concise sustainability reports.

Reports should include:
- Key emissions KPIs
- Carbon intensity in kg CO2e per unit
- Performance against targets
- Short, actionable recommendations

Use tables for comparisons and avoid unnecessary
technical jargon.
"""
    }
]

for memory in memories:
    response = requests.post(
        memory_url,
        headers=headers,
        params={"scope": MEMORY_SCOPE},
        json=memory,
        timeout=30
    )

    print(memory["path"], response.status_code)
    response.raise_for_status()

print("All CarbonOps memories stored successfully!")

### Define the User Query

In [ ]:
user_question = (
    "How is our Ahmedabad factory performing "
    "against its sustainability targets?"
)

print("USER:", user_question)

### Retrieve the Relevant Managed Memories

In [ ]:
search_url = (
    f"{workspace_host}/api/2.1/unity-catalog/"
    f"memory-stores/{MEMORY_STORE}/entries:search"
)

payload = {
    "scope": MEMORY_SCOPE,
    "query": user_question,
    "top_k": 5
}

response = requests.post(
    search_url,
    headers=headers,
    json=payload,
    timeout=30
)

response.raise_for_status()

search_results = response.json()

print(search_results)

In [ ]:
results = search_results.get("results", [])

memory_context = "\n\n".join(
    result["memory_entry"]["contents"]
    for result in results
    if result.get("memory_entry", {}).get("contents")
)

print("=== RETRIEVED MEMORY ===")
print(memory_context)

### Send Memory Context with the LLM API Call

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ChatMessage,
    ChatMessageRole
)

w = WorkspaceClient(
    host=workspace_host,
    token=DATABRICKS_TOKEN
)

user_prompt = f"""
You are the CarbonOps Sustainability Intelligence Assistant.

Use the following retrieved long-term memory to answer
the sustainability manager's question.

--- RETRIEVED MEMORY ---
{memory_context}
--- END MEMORY ---

Instructions:
- Use relevant information from memory.
- Calculate carbon intensity when sufficient data exists.
- Compare performance against the remembered targets.
- Provide concise and actionable recommendations.
- Treat remembered figures as historical or illustrative
  unless they are explicitly confirmed as current.
- Do not invent missing operational data.

User question:
{user_question}
"""

response = w.serving_endpoints.query(
    name="databricks-claude-sonnet-4-5",
    messages=[
        ChatMessage(
            role=ChatMessageRole.SYSTEM,
            content="You are a helpful sustainability assistant."
        ),
        ChatMessage(
            role=ChatMessageRole.USER,
            content=user_prompt
        )
    ],
    max_tokens=2048
)

print("=== CARBONOPS ASSISTANT RESPONSE ===")
print(response.choices[0].message.content)